In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount = False)
!mkdir -p /content/vistext/  # Create the target directory if it doesn't exist
!ln -s -f /content/drive/MyDrive/EvaltheEvaluators/* /content/vistext/

import pandas as pd
# Load the saved charts_df
charts_df = pd.read_csv('/content/drive/MyDrive/EvaltheEvaluators/charts_df.csv')

from google.colab import userdata
HF_TOKEN =userdata.get('HF_TOKEN')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
drive_target_path = '/content/drive/MyDrive/EvaltheEvaluators'

print(f"Contents of {drive_target_path}:")
if os.path.exists(drive_target_path):
    for item in os.listdir(drive_target_path):
        print(item)
else:
    print(f"The directory {drive_target_path} does not exist.")

Contents of /content/drive/MyDrive/EvaltheEvaluators:
data
charts_df.csv
vistext
images
ChartGemma_Model


In [ ]:
!pip install --break-system-packages transformers torch pillow bitsandbytes accelerate --quiet
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from PIL import Image
import torch
import json
import glob

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = AutoProcessor.from_pretrained('Qwen/Qwen2-VL-2B-Instruct')

model = Qwen2VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen2-VL-2B-Instruct',
    torch_dtype = torch.float16,
    device_map = "auto"
)
model.eval()
device = model.device
Qwen_Descriptions = []
batch_size = 1 # for 2B model

Using device: cuda


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

In [ ]:
## 1. Initialise List of Chart IDs and Question
question = "Describe the chart in detail. First, state the starting value, the ending value, and any peak or minimum values. Then, summarize the overall visual trend and the rate of change. Do not speculate on external real-world causes; rely strictly on the data shown. Write your response as a cohesive paragraph of 3-5 sentences.Constraint: Your final response must be strictly between 50 and 100 words."
chart_ids = [os.path.basename(f).replace('.png', '') for f in glob.glob(f'{'/content/vistext/images/'}*.png')]

# Process in batches
with torch.no_grad():
    for batch_idx in range(0, len(chart_ids), batch_size):
        batch_ids = chart_ids[batch_idx:batch_idx + batch_size]
        batch_num = (batch_idx // batch_size) + 1
        images = []
        valid_ids = []

        ## 2. Load images for this batch
        for img_id in batch_ids:
            image_path = f'{'/content/vistext/images/'}{img_id}.png'
            try:
                image = Image.open(image_path).convert('RGB')
                images.append(image)
                valid_ids.append(img_id)
            except Exception as e:
                print(f"Error loading image {img_id}: {e}")
        if not images:
            print(f"Batch {batch_num}: No valid images, skipping...")
            continue

        ## 3. Format inputs for Qwen
        try:
            # Qwen expects a list of messages (a conversation) for EACH item
            conversations = [[{
                        "role": "user",
                        "content": [
                            {"type": "image", "image": img},
                            {"type": "text", "text": question},
                                   ],}]for img in images]
            texts = [processor.apply_chat_template(conv,
                                              tokenize = False,
                                              add_generation_prompt = True)
                for conv in conversations]
            inputs = processor(
                text = texts,
                images = images,
                return_tensors = "pt",
                padding = True
            )
            # move to device
            inputs = {k: v.to(device) if hasattr(v, 'to') else v for k, v in inputs.items()}

            ## 4. generate predictions for batch
            output_ids = model.generate(
                **inputs,
                max_new_tokens = 200, # Stops it from rambling past ~150 words
                repetition_penalty = 1.2,
                do_sample = False
            )
            ## 5. Decode results
            generated_ids = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs['input_ids'], output_ids)]
            answers = processor.batch_decode(generated_ids, skip_special_tokens=True)
            for idx, (img_id, answer) in enumerate(zip(valid_ids, answers)):
                answer = answer.strip()
                print(f"({batch_idx + idx + 1}/{len(chart_ids)}) Image ID: {img_id}")
                print(f"Answer: {answer}\n")
                Qwen_Descriptions.append({
                    'img_id': img_id,
                    'answer': answer
                })
        except Exception as e2:
            print(f"Error processing batch {batch_num}: {e2}\n")

        ## 7. Clear cache between batches
        torch.cuda.empty_cache()

## 8. Save results
os.makedirs('Qwen_Model', exist_ok=True)
with open('Qwen_Model/Qwen_Descriptions.json', 'w') as f:
    json.dump(Qwen_Descriptions, f, indent=2)

print(f"\n{'-'*20}")
print(f"Processed {len(Qwen_Descriptions)}/{len(chart_ids)} charts successfully")
print(f"Results saved to: Qwen_Model/Qwen_Descriptions.json")
print(f"{'-'*20}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


(1/21) Image ID: 3128
Answer: The graph depicts the infant mortality rate from 2009 to 2018 for Armenia (in deaths per 1,000 live births). The starting point is at approximately 16 deaths per 1,000 live births, which gradually decreases over time until it reaches around 7 by 2018. There's no significant increase or decrease observed during this period. Overall, there seems to have been a steady decline in the infant mortality rate within the given timeframe.

(2/21) Image ID: 2114
Answer: The graph titled "Iran : Youth unemployment rate from 1999 to 2020" depicts the youth unemployment rates over several years for Iran. The x-axis represents the year while the y-axis shows the youth unemployment rate.

Starting with an initial value around 0.24 at the beginning of the period (around 2000), there is a gradual increase up until about 2008 when it reaches its highest point near 0.30. After that, there's a slight decline but remains above 0.26 throughout most of the subsequent years. There

In [ ]:
import os
import json

# Ensure the Qwen_Model directory exists
os.makedirs('Qwen_Model', exist_ok=True)

# Save the Qwen_Descriptions to a JSON file
with open('Qwen_Model/Qwen_Descriptions.json', 'w') as f:
    json.dump(Qwen_Descriptions, f, indent=2)

In [ ]:
import shutil
import os

source_path = 'Qwen_Model'
target_path = '/content/drive/MyDrive/EvaltheEvaluators/Qwen_Model'

# Ensure the source exists
if os.path.exists(source_path):
    # Ensure the target directory exists on Google Drive
    os.makedirs(target_path, exist_ok=True)
    !cp -r {source_path}/* {target_path}/
    # remove local files if copy was successful
    shutil.rmtree(source_path)
else:
    print(f"Source directory {source_path} not found.")